<a href="https://colab.research.google.com/github/Pranajit04/Machine-Learning-Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pranajit04/Machine-Learning-Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Signal checks

Two signals my rule leans on, checked before I trust them.

In [12]:
%pip -q install pandas
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv')

# Signal 1: staleness (behind the real refresh flag)
df['age_bucket'] = pd.cut(df['content_age_days'], bins=[0,90,365,730,99999],
                            labels=['0-90d','90-365d','1-2yr','2yr+'])
bucket_table_1 = df.groupby('age_bucket')['trend_direction'].value_counts(normalize=True).unstack()
bucket_table_1['n'] = df.groupby('age_bucket').size()
print("Signal 1 — staleness vs trend:")
print(bucket_table_1)
print()

# Signal 2: CTR vs position (behind the real CTR-fix flag)
df['position_bucket'] = pd.cut(df['avg_position'], bins=[0,10,20,30,100],
                                 labels=['top10','11-20','21-30','30+'])
bucket_table_2 = df.groupby('position_bucket')['ctr'].agg(['mean','count'])
bucket_table_2.columns = ['avg_ctr','n']
print("Signal 2 — position vs CTR:")
print(bucket_table_2)

Signal 1 — staleness vs trend:
trend_direction      down      flat       new    stable        up      n
age_bucket                                                              
0-90d            0.668699  0.024390  0.036585  0.117886  0.152439    492
90-365d          0.571194  0.043114  0.091023  0.170425  0.124244  23148
1-2yr            0.426258  0.022327  0.017453  0.308019  0.225943   6360
2yr+             0.000000  0.000000  0.000000  0.000000  0.000000      0

Signal 2 — position vs CTR:
                  avg_ctr      n
position_bucket                 
top10            0.832373  12983
11-20            0.323443   7273
21-30            0.308960   3933
30+              0.128388   4591


/tmp/ipykernel_1060/3174502379.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_table_1 = df.groupby('age_bucket')['trend_direction'].value_counts(normalize=True).unstack()
/tmp/ipykernel_1060/3174502379.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_table_1['n'] = df.groupby('age_bucket').size()
/tmp/ipykernel_1060/3174502379.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_table_2 = df

**Signal 1 verdict: OPPOSITE.** Newer content (0-90d) declines 66.9% of the
time, vs 42.6% for 1-2yr content — the reverse of the staleness hypothesis.
Note: the 2yr+ bucket is empty (n=0), a real gap in this dataset. Because
this came back opposite, I do NOT use content_age_days as a decline trigger
in my rule below — this negative result saved me from building a rule on
a false assumption.

**Signal 2 verdict: CONFIRMED.** avg_ctr drops monotonically as position
worsens (83.2% → 32.3% → 30.9% → 12.8%), directly matching the session's
CTR-fix logic. This signal is trustworthy and drives my rule.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule (plain words):** flag content as needing a refresh if it's
currently declining (trend_direction == down), ranked by search volume so
the biggest-impact pages surface first. Flag as needing a CTR fix if it
ranks poorly (position > 20) but still has real search volume. Flag as a
quick win if it's high-volume and already stable or growing. Staleness
(content age) is deliberately NOT used as a trigger, since Signal 1 showed
it points the wrong direction in this data.

**Reason codes:** declining_needs_refresh, poor_position_low_ctr, high_volume_stable, healthy

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quick check: confirm the rule's inputs match what the signal checks validated
print(df[['trend_direction','avg_position','ctr','search_volume']].describe(include='all'))


       trend_direction  avg_position           ctr  search_volume
count            30000   30000.00000  30000.000000   27532.000000
unique               5           NaN           NaN            NaN
top               down           NaN           NaN            NaN
freq             16262           NaN           NaN            NaN
mean               NaN      16.34238      0.510733     158.882391
std                NaN      15.21679      3.279162    1518.270825
min                NaN       0.00000      0.000000       0.000000
25%                NaN       6.20000      0.000000       0.000000
50%                NaN      10.80000      0.070000      10.000000
75%                NaN      22.30000      0.290000      20.000000
max                NaN     245.00000    100.000000   74000.000000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Coding the rule from Section 1 into a score, ranking every row, and writing
the CSV. Score priority: declining pages first (biggest volume first),
then poor-position/low-CTR pages, then quick wins.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

def score_and_flag(row):
    score = 0
    reason = "healthy"
    action = "monitor"

    if row['trend_direction'] == 'down':
        score = 3 if row['search_volume'] > df['search_volume'].median() else 2
        reason = "declining_needs_refresh"
        action = "refresh"
    elif row['avg_position'] > 20 and row['ctr'] < df['ctr'].median():
        score = 2
        reason = "poor_position_low_ctr"
        action = "ctr_fix"
    elif row['search_volume'] > df['search_volume'].quantile(0.75) and row['trend_direction'] in ['up','stable']:
        score = 1
        reason = "high_volume_stable"
        action = "quick_win"

    return pd.Series([score, reason, action])

df[['action_score','reason_code','action_label']] = df.apply(score_and_flag, axis=1)

queue = df.sort_values('action_score', ascending=False)[
    ['content_id','client_id','action_score','reason_code','action_label','search_volume','avg_position','ctr']
]

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")
queue.head(20)


Wrote 30000 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,action_score,reason_code,action_label,search_volume,avg_position,ctr
18,content_0b360eb9db55,client_349c41201b,3,declining_needs_refresh,refresh,30.0,11.4,0.14
19,content_af865035b328,client_f369cb89fc,3,declining_needs_refresh,refresh,30.0,6.9,2.02
20694,content_839755102d3f,client_3fdba35f04,3,declining_needs_refresh,refresh,170.0,26.3,0.00
20695,content_433e980f7076,client_3fdba35f04,3,declining_needs_refresh,refresh,590.0,54.6,0.00
25,content_033ae3e7aecf,client_f369cb89fc,3,declining_needs_refresh,refresh,70.0,7.2,0.00
29988,content_9bb9a0584cae,client_8527a891e2,3,declining_needs_refresh,refresh,210.0,54.0,0.00
29994,content_995627a1f490,client_7f2253d7e2,3,declining_needs_refresh,refresh,70.0,4.6,0.86
21521,content_8573511e9f28,client_4e07408562,3,declining_needs_refresh,refresh,30.0,38.7,0.04
29966,content_77867ed726e1,client_19581e27de,3,declining_needs_refresh,refresh,30.0,12.7,0.08
60,content_b9104a222d01,client_f369cb89fc,3,declining_needs_refresh,refresh,30.0,6.2,4.00


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20)
for i, row in top20.iterrows():
    print(f"{row['content_id']} — {row['action_label']} — {row['reason_code']} — vol:{row['search_volume']} pos:{row['avg_position']:.1f} ctr:{row['ctr']:.3f}")


content_0b360eb9db55 — refresh — declining_needs_refresh — vol:30.0 pos:11.4 ctr:0.140
content_af865035b328 — refresh — declining_needs_refresh — vol:30.0 pos:6.9 ctr:2.020
content_839755102d3f — refresh — declining_needs_refresh — vol:170.0 pos:26.3 ctr:0.000
content_433e980f7076 — refresh — declining_needs_refresh — vol:590.0 pos:54.6 ctr:0.000
content_033ae3e7aecf — refresh — declining_needs_refresh — vol:70.0 pos:7.2 ctr:0.000
content_9bb9a0584cae — refresh — declining_needs_refresh — vol:210.0 pos:54.0 ctr:0.000
content_995627a1f490 — refresh — declining_needs_refresh — vol:70.0 pos:4.6 ctr:0.860
content_8573511e9f28 — refresh — declining_needs_refresh — vol:30.0 pos:38.7 ctr:0.040
content_77867ed726e1 — refresh — declining_needs_refresh — vol:30.0 pos:12.7 ctr:0.080
content_b9104a222d01 — refresh — declining_needs_refresh — vol:30.0 pos:6.2 ctr:4.000
content_d5d3c2e98937 — refresh — declining_needs_refresh — vol:40.0 pos:28.9 ctr:0.000
content_f10930228dff — refresh — declining_n

1. content_0b360eb9db55 — refresh, declining_needs_refresh. Vol 30, pos 11.4, ctr 0.14. Wrong if: this is a niche page where 30/mo volume is already near its natural ceiling.
2. content_af865035b328 — refresh, declining_needs_refresh. Vol 30, pos 6.9, ctr 2.02. Wrong if: ctr 2.02 seems oddly high for pos ~7 (top10 avg was ~83%) — may be a data quirk worth checking.
3. content_839755102d3f — refresh, declining_needs_refresh. Vol 170, pos 26.3, ctr 0.00. Wrong if: 0.00 ctr with real volume suggests a tracking gap, not true decline.
4. content_433e980f7076 — refresh, declining_needs_refresh. Vol 590, pos 54.6, ctr 0.00. Wrong if: position 54.6 is so poor this page may never have realistically ranked — refresh may not fix that alone.
5. content_033ae3e7aecf — refresh, declining_needs_refresh. Vol 70, pos 7.2, ctr 0.00. Wrong if: good position (7.2) with 0.00 ctr is suspicious — likely a snippet/metadata issue, not a content-freshness issue.
6. content_9bb9a0584cae — refresh, declining_needs_refresh. Vol 210, pos 54.0, ctr 0.00. Wrong if: same as #4 — poor position may be the real root cause, not staleness.
7. content_995627a1f490 — refresh, declining_needs_refresh. Vol 70, pos 4.6, ctr 0.86. Wrong if: position 4.6 with real CTR looks healthy on the surface — declining trend may be temporary, not structural.
8. content_8573511e9f28 — refresh, declining_needs_refresh. Vol 30, pos 38.7, ctr 0.04. Wrong if: this is a genuinely low-priority page not worth review time given tiny volume.
9. content_77867ed726e1 — refresh, declining_needs_refresh. Vol 30, pos 12.7, ctr 0.08. Wrong if: same — low volume may not justify action.
10. content_b9104a222d01 — refresh, declining_needs_refresh. Vol 30, pos 6.2, ctr 4.00. Wrong if: ctr 4.00 with good position looks like it's actually performing well — declining trend might be noise.
11. content_d5d3c2e98937 — refresh, declining_needs_refresh. Vol 40, pos 28.9, ctr 0.00. Wrong if: low volume, may not be worth reviewer time.
12. content_f10930228dff — refresh, declining_needs_refresh. Vol 30, pos 42.9, ctr 0.00. Wrong if: position this poor may need a different fix than a content refresh.
13. content_8b08ec7fc725 — refresh, declining_needs_refresh. Vol 20, pos 7.3, ctr 0.00. Wrong if: good position + zero ctr again points to metadata, not content quality.
14. content_7b84d6a08064 — refresh, declining_needs_refresh. Vol 20, pos 10.5, ctr 0.09. Wrong if: very low volume (20) — impact of fixing this is minimal either way.
15. content_19882bd6373d — refresh, declining_needs_refresh. Vol 30, pos 9.0, ctr 0.30. Wrong if: decent ctr for this position — may already be performing reasonably.
16. content_e76ac7fed711 — refresh, declining_needs_refresh. Vol 260, pos 5.8, ctr 0.05. Wrong if: excellent position (5.8) with tiny ctr is very likely a metadata/snippet problem, not staleness.
17. content_696e802c58d6 — refresh, declining_needs_refresh. Vol 50, pos 39.8, ctr 0.00. Wrong if: position this bad may need a ranking fix before a content refresh matters.
18. content_44fd64e054b3 — refresh, declining_needs_refresh. Vol 260, pos 16.5, ctr 0.12. Wrong if: this one looks like a fair candidate — real volume, mid position, low ctr; refresh is a reasonable action here.
19. content_62ece016c9d8 — refresh, declining_needs_refresh. Vol 20, pos 9.4, ctr 0.00. Wrong if: tiny volume, likely low priority despite the flag.
20. content_1c78cae469ad — refresh, declining_needs_refresh. Vol 20, pos 8.5, ctr 0.27. Wrong if: small volume again limits real impact.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak pick pattern (real, honest finding):** every single row in my top 20
has the exact same score (3) and same action (refresh) — the queue sorted
only by score with no tiebreaker, so among thousands of tied rows, the
order shown is essentially arbitrary. Several "top 20" picks have tiny
search_volume (20-30), contradicting my own rule's stated goal of
"biggest-impact pages surface first." A real fix: add a secondary sort by
search_volume within each score tier.

Several rows (e.g. #3, #5, #13, #16) show strong position (single digits)
paired with 0.00 ctr — that pattern looks more like a metadata/snippet
problem than genuine content decline, which the rule can't currently tell
apart. Worth a reason-code split in a future version.

**Leakage check:** no future-window or label-derived columns were used as
rule inputs — trend_direction, avg_position, ctr, and search_volume are all
observed, already-known signals, not derived from any later time window or
outcome the rule is trying to predict.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Fix the tiebreak issue found above, and re-verify no leakage columns are used
queue_fixed = df.sort_values(['action_score','search_volume'], ascending=[False,False])[
    ['content_id','client_id','action_score','reason_code','action_label','search_volume','avg_position','ctr']
]
queue_fixed.to_csv('work/outputs/baseline_action_score.csv', index=False)
print("Overwrote CSV with tiebreak-fixed queue")

print("Top 5 after adding search_volume as tiebreaker:")
print(queue_fixed.head())

rule_inputs = ['trend_direction','avg_position','ctr','search_volume']
print("\nRule inputs used:", rule_inputs)
print("Any future-window columns used?", any(c in rule_inputs for c in ['impressions_last_30d','clicks_last_30d','trend_pct']))

Overwrote CSV with tiebreak-fixed queue
Top 5 after adding search_volume as tiebreaker:
                 content_id          client_id  action_score  \
6972   content_bf67a444faef  client_3fdba35f04             3   
28282  content_454cc6654c6e  client_3fdba35f04             3   
8055   content_cd6760921db8  client_3fdba35f04             3   
13502  content_f76ccf7a7834  client_19581e27de             3   
22788  content_ee4630879d03  client_3fdba35f04             3   

                   reason_code action_label  search_volume  avg_position   ctr  
6972   declining_needs_refresh      refresh        60500.0          45.5  0.00  
28282  declining_needs_refresh      refresh        60500.0          44.9  0.00  
8055   declining_needs_refresh      refresh        49500.0          47.3  0.00  
13502  declining_needs_refresh      refresh        49500.0           9.5  0.15  
22788  declining_needs_refresh      refresh        49500.0          25.2  0.00  

Rule inputs used: ['trend_direction', 'a

**Self Check**

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.